# DAY 11


In [17]:
# Yeh direct GEE ke system tool ko bypass mode me run karega
!earthengine authenticate --quiet

Authenticate: Limited support in Colab. Use ee.Authenticate() or --auth_mode=notebook instead.
Authenticate: Credentials already exist. Use --force to refresh.


In [18]:
import ee
import pprint

# 1. GEE ko tumhari Cloud Project ID ke sath initialize karna
ee.Initialize(project='iron-airfoil-496807-j7')

# 2. Odisha ka Region of Interest (ROI)
odisha_roi = ee.Geometry.Polygon([
    [[84.0, 19.0], [87.0, 19.0], [87.0, 22.0], [84.0, 22.0]]
])

# 3. Sentinel-2 ke badal (clouds) saaf karne ka corrected function
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # GEE me logical AND ke liye .And() use hota hai capital 'A' ke sath
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))

    return image.updateMask(mask).divide(10000)

# 4. Sentinel-2 Data pull karna aur filter karna
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                 .filterBounds(odisha_roi)
                 .filterDate('2025-01-01', '2026-05-01')
                 .map(mask_s2_clouds))

# Poore saal ka ek clean mosaic composite banana
composite_image = s2_collection.median()

# 5. NDVI Calculate karna (NIR = B8, RED = B4)
ndvi = composite_image.expression(
    '(NIR - RED) / (NIR + RED)', {
        'NIR': composite_image.select('B8'),
        'RED': composite_image.select('B4')
    }
).rename('NDVI_Industrial')

# 6. Output verify karne ke liye band info print karna
print("--- CONGRATULATIONS BHAI, FINALLY DONE ---")
pprint.pprint(ndvi.getInfo()['bands'])

--- CONGRATULATIONS BHAI, FINALLY DONE ---
[{'crs': 'EPSG:4326',
  'crs_transform': [1, 0, 0, 0, 1, 0],
  'data_type': {'precision': 'float', 'type': 'PixelType'},
  'id': 'NDVI_Industrial'}]


In [19]:
# Image ko Google Drive me bhejte hain, scale 10 meters rakhenge kyunki Sentinel-2 hai
task = ee.batch.Export.image.toDrive(
    image=ndvi,
    description='Odisha_NDVI_Industrial_Map',
    folder='GEE_Day10_Outputs',       # Yeh folder aapke Drive me automatic ban jayega
    fileNamePrefix='Odisha_NDVI_2026',
    scale=500,                           # 500-meter resolution for best quality
    region=odisha_roi,                  # Odisha ka boundary box जो ऊपर define kiya tha
    fileFormat='GeoTIFF',
    maxPixels=1e13                      # Taaki pixels limit exceed na ho
)

# Export task ko start karna
task.start()

print("--- EXPORT TASK STARTED SUCCESSFULY ---")
print("Bhai, task shuru ho gaya hai! Status check karne ke liye niche wala cell run karo.")

--- EXPORT TASK STARTED SUCCESSFULY ---
Bhai, task shuru ho gaya hai! Status check karne ke liye niche wala cell run karo.


In [20]:
import time

# Yeh code har 15 second me status check karke batayega
while task.active():
    print(f"Exporting status: {task.status()['state']}... (Thoda sabr rakho bhai)")
    time.sleep(15)

print(f"Final Status: {task.status()['state']}")
if task.status()['state'] == 'COMPLETED':
    print("Mubarak ho bhai! Google Drive me 'GEE_Day10_Outputs' naam ka folder check karo, file vahan pahonch gayi hai!")
else:
    print(f"Kuch gadbad hui lagta hai: {task.status().get('error_message', 'Unknown Error')}")

Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Thoda sabr rakho bhai)
Exporting status: READY... (Tho